# 14A — OpenElections contribution correlations

## Main question

**Which dimensions of candidates' fundraising profiles are most strongly
associated with ballot support?**

We compare many candidate-level X variables with two electoral outcomes:

- `mentions`
- `first_place_votes`

The notebook has three parts:

**A. Prepare OpenElections**  
**B. Build all X variables**  
**C. Run and rank correlations**

> These are descriptive associations, not causal effects.


# 0. Map of the X variables

Before running the analysis, this section explains exactly what kinds of
candidate features we are creating.

## Families

| Family | What it captures | Example |
|---|---|---|
| `scale` | Overall size of the candidate's fundraising | `total_amount` |
| `overall_bins` | Overall contribution-size profile | `bin_count_small` |
| `cash_in_kind` | Cash vs. In-kind composition | `cash_amount_share` |
| `cash_in_kind_bins` | Contribution-size profile within Cash or In-kind | `cash_bin_count_share_small` |
| `individual_business` | Individual vs. Business composition | `individual_count_share` |
| `individual_business_bins` | Contribution-size profile within Individual or Business | `individual_bin_amount_share_small` |
| `geography` | Where contribution records come from | `same_district_share` |
| `geography_bins` | Contribution-size profile within a geography | `portland_bin_count_share_small` |

## Measures

There are four main ways to describe a group or bin:

| Measure | Meaning |
|---|---|
| `dollar amount` | Total dollars in the group/bin |
| `dollar share` | Share of relevant dollars in the group/bin |
| `record count` | Number of contribution records in the group/bin |
| `record share` | Share of relevant contribution records in the group/bin |

We use **contribution record**, not unique donor. OpenElections does not give
us a unique donor ID for this analysis.


## How the five bins are constructed

Every contribution record is assigned to a bin using its individual dollar
amount:

| Bin | Contribution amount |
|---|---:|
| `micro` | <= $25 |
| `small` | > $25 and <= $100 |
| `medium` | > $100 and <= $250 |
| `large` | > $250 and <= $1,000 |
| `mega` | > $1,000 |

The bin itself is always based on **transaction amount**.

After that, we can describe the same bin at two different levels.

Example for Small contributions:

```text
bin_count_small
    = number of Small contribution records

bin_count_share_small
    = Small records / all contribution records

bin_amount_small
    = dollars raised through Small contributions

bin_amount_share_small
    = Small-contribution dollars / total fundraising dollars
```


## Which measures exist in each family?

To keep the analysis interpretable, we do **not** create every possible
combination mechanically.

| Family | Dollar amount | Dollar share | Record count | Record share |
|---|:---:|:---:|:---:|:---:|
| Scale | x | — | x | — |
| Overall bins | x | x | x | x |
| Cash / In-kind overall | x | x | x | x |
| Cash / In-kind × bins | x | x | x | x |
| Individual / Business overall | x | x | x | x |
| Individual / Business × bins | x | x | x | x |
| Geography overall | — | — | x | x |
| Geography × bins | — | — | x | x |

Geography is intentionally **count-based for now**.

The geographic shares use explicit eligible universes:

- own-district share → records with usable coordinates;
- Portland share → records with usable coordinates;
- Oregon share → records with a reported state.


## 1. Setup

In [31]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

from IPython.display import display


cwd = Path.cwd().resolve()


if (cwd / "pyproject.toml").exists():
    ROOT = cwd

elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )


if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT),
    )


from helpers.paths import (
    CLEAN,
    PROCESSED,
    RAW,
)


from helpers.finance_feature_helpers import (
    BIN_LABELS,
    add_amount_bin,
    build_base_features,
    add_bin_features,
    add_group_features,
    add_group_bin_features,
    add_flag_features,
    add_flag_bin_features,
    calculate_correlations,
    calculate_correlations_by_district,
    make_correlation_race,
    make_district_race,
    compare_top_features_across_districts,
)


pd.set_option(
    "display.max_columns",
    200,
)


print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


# 2. CHANGE ONLY THIS CELL

`DISTRICT` controls the **main correlation race**.

```python
YEAR = 2024
DISTRICT = "all"
```

means:

> use all 2024 candidates in the main race.

```python
YEAR = 2024
DISTRICT = 4
```

means:

> use only District 4 candidates in the main race.

### Important

The **district races are always calculated independently for D1, D2, D3, and D4**.

So even if:

```python
DISTRICT = 4
```

the later district section will still calculate separate races for all four districts.


In [32]:
YEAR = 2024

# Options: "all", 1, 2, 3, 4
DISTRICT = "all"


OUTCOMES = [
    "mentions",
    "first_place_votes",
]


# ---------------------------------------------------------------
# Display settings
# ---------------------------------------------------------------

# Number of X variables shown in the pooled correlation race.
TOP_N = 20

# Minimum candidate sample required for a district correlation.
MIN_DISTRICT_N = 8

# Number of X variables shown in each district race.
DISTRICT_TOP_N = 15

# Number of pooled winners compared across D1-D4.
COMPARE_TOP_N = 15


## 3. Canonical paths inside the repo

In [33]:
CONTRIBUTIONS_PATH = (
    CLEAN
    / "portland_contributions"
    / "contributions.csv"
)


MASTER_DIR = (
    PROCESSED
    / "master"
)


BALLOT_SUPPORT_DIR = (
    PROCESSED
    / "ballot_support"
)


PRECINCT_SHAPEFILE = (
    RAW
    / "report2025"
    / "shapefiles"
    / "Portland precincts with pop"
    / "Portland_precincts_repaired_with_pop.shp"
)


for path in [
    CONTRIBUTIONS_PATH,
    PRECINCT_SHAPEFILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required repo file: {path}"
        )


print("Contributions:", CONTRIBUTIONS_PATH)
print("Precinct shapefile:", PRECINCT_SHAPEFILE)


Contributions: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/portland_contributions/contributions.csv
Precinct shapefile: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/raw/report2025/shapefiles/Portland precincts with pop/Portland_precincts_repaired_with_pop.shp


# PART A — PREPARE OPENELECTIONS

## 4. Load ballot support

In [34]:
support_files = sorted(
    BALLOT_SUPPORT_DIR.glob(
        "*/candidate_ballot_support_*.csv"
    )
)


support_parts = []


for path in support_files:

    part = pd.read_csv(
        path,
        low_memory=False,
    )

    support_parts.append(
        part
    )


support = pd.concat(
    support_parts,
    ignore_index=True,
    sort=False,
)


support["year"] = pd.to_numeric(
    support["year"],
    errors="coerce",
).astype("Int64")


support["district"] = pd.to_numeric(
    support["district"],
    errors="coerce",
).astype("Int64")


if YEAR is not None:

    support = support[
        support["year"].eq(
            YEAR
        )
    ].copy()

if support.empty:

    raise ValueError(
        "No ballot-support rows remain after filtering."
    )


SELECTED_YEARS = sorted(
    support["year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


print("Selected years:", SELECTED_YEARS)
print(
    "Main analysis district:",
    DISTRICT,
)
print(
    "Candidates:",
    len(support),
)


Selected years: [2024]
Main analysis district: all
Candidates: 98


## 5. Load and filter OpenElections contributions

In [35]:
contributions = pd.read_csv(
    CONTRIBUTIONS_PATH,
    low_memory=False,
)


finance = contributions.copy()


finance["amount"] = pd.to_numeric(
    finance["amount"],
    errors="coerce",
)


finance["year"] = pd.to_numeric(
    finance["year"],
    errors="coerce",
).astype("Int64")


finance["district"] = pd.to_numeric(
    finance["district"],
    errors="coerce",
).astype("Int64")


finance = finance[
    finance["year"].isin(
        SELECTED_YEARS
    )
].copy()

if "is_public_matching_contribution" in finance.columns:

    public_matching = (
        finance[
            "is_public_matching_contribution"
        ]
        .astype("string")
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )

else:

    public_matching = pd.Series(
        False,
        index=finance.index,
    )


finance = finance[
    ~public_matching
    & finance["amount"].notna()
    & finance["amount"].gt(0)
].copy()


print(
    "Private positive contribution records:",
    f"{len(finance):,}",
)

print(
    "Private fundraising:",
    f"${finance['amount'].sum():,.2f}",
)


Private positive contribution records: 43,110
Private fundraising: $2,077,989.10


## 6. Link finance records to official candidates

In [36]:
crosswalk_parts = []


for year in SELECTED_YEARS:

    path = (
        MASTER_DIR
        / f"candidate_source_crosswalk_{year}.csv"
    )


    crosswalk = pd.read_csv(
        path,
        low_memory=False,
    )


    crosswalk = crosswalk[
        crosswalk["source"].eq(
            "portland_contributions"
        )
        & crosswalk["classification"].eq(
            "match"
        )
    ].copy()


    crosswalk_parts.append(
        crosswalk
    )


finance_crosswalk = pd.concat(
    crosswalk_parts,
    ignore_index=True,
)


matched_names = (
    finance_crosswalk[
        [
            "year",
            "district",
            "source_candidate_name",
            "suggested_candidate",
            "suggested_candidate_key",
        ]
    ]
    .rename(
        columns={
            "suggested_candidate":
                "canonical_candidate",
            "suggested_candidate_key":
                "candidate_key",
        }
    )
    .drop_duplicates()
)


finance = finance.merge(
    matched_names,
    left_on=[
        "year",
        "district",
        "candidate",
    ],
    right_on=[
        "year",
        "district",
        "source_candidate_name",
    ],
    how="left",
    validate="many_to_one",
)


unmatched = (
    finance[
        finance["candidate_key"].isna()
    ]
    .groupby(
        [
            "year",
            "district",
            "candidate",
        ],
        as_index=False,
    )
    .agg(
        records=(
            "amount",
            "size",
        ),
        total_amount=(
            "amount",
            "sum",
        ),
    )
    .sort_values(
        "total_amount",
        ascending=False,
    )
)


print(
    "Unmatched source labels:",
    len(unmatched),
)


if len(unmatched) > 0:

    display(
        unmatched
    )


finance = finance[
    finance["candidate_key"].notna()
].copy()


PROFILE_KEYS = [
    "year",
    "district",
    "candidate_key",
]


candidate_names = (
    finance[
        PROFILE_KEYS
        + [
            "canonical_candidate",
        ]
    ]
    .drop_duplicates()
)


Unmatched source labels: 4


,year,district,candidate,records,total_amount
2,2024,2,John Middleton,42,5005.0
0,2024,1,Sonja McKenzie,75,4214.0
1,2024,2,David Burnell,45,1943.0
3,2024,3,Robin Ye,4,100.0


## 7. Inspect source categories before recoding

These tables stay visible because Cash/In-kind and Individual/Business are
methodological decisions, not merely programming details.


In [37]:
print("contributionSubType")

display(
    finance[
        "contributionSubType"
    ]
    .fillna("<missing>")
    .value_counts(
        dropna=False
    )
    .to_frame(
        "records"
    )
)


print()
print("contributorType")

display(
    finance[
        "contributorType"
    ]
    .fillna("<missing>")
    .value_counts(
        dropna=False
    )
    .to_frame(
        "records"
    )
)


contributionSubType


,records
contributionSubType,
cash,42487
item_sold_fair_market,194
inkind_contribution,145
item_misc,58
item_refund,44
item_returned_check,14
inkind_forgiven_personal,2



contributorType


,records
contributorType,
individual,42527
business,111
political_committee,109
other,78
family,56
political_party,40
labor,16
unregistered,7


## 8. Recode Cash/In-kind and Individual/Business

In [38]:
subtype = (
    finance[
        "contributionSubType"
    ]
    .astype("string")
    .str.strip()
    .str.casefold()
)


finance["subtype_group"] = "other"


finance.loc[
    subtype.eq("cash"),
    "subtype_group",
] = "cash"


finance.loc[
    subtype.str.startswith(
        "inkind",
        na=False,
    ),
    "subtype_group",
] = "in_kind"


entity = (
    finance[
        "contributorType"
    ]
    .astype("string")
    .str.strip()
    .str.casefold()
)


finance["entity_group"] = "other"


finance.loc[
    entity.eq("individual"),
    "entity_group",
] = "individual"


finance.loc[
    entity.eq("business"),
    "entity_group",
] = "business"


print("Recoded subtype")

display(
    finance[
        "subtype_group"
    ]
    .value_counts()
    .to_frame(
        "records"
    )
)


print()
print("Recoded entity type")

display(
    finance[
        "entity_group"
    ]
    .value_counts()
    .to_frame(
        "records"
    )
)


Recoded subtype


,records
subtype_group,
cash,42487
other,310
in_kind,147



Recoded entity type


,records
entity_group,
individual,42527
other,306
business,111


## 9. Add the five transaction-amount bins

In [39]:
finance = add_amount_bin(
    finance,
    amount_column="amount",
    bin_column="amount_bin",
)


display(
    finance[
        "amount_bin"
    ]
    .value_counts(
        sort=False
    )
    .to_frame(
        "records"
    )
)


,records
amount_bin,
micro,29844
small,9944
medium,1528
large,1620
mega,8


## 10. Build geographic flags

In [40]:
precincts = gpd.read_file(
    PRECINCT_SHAPEFILE
)


precincts["District"] = pd.to_numeric(
    precincts["District"],
    errors="coerce",
)


council_districts = (
    precincts
    .dropna(
        subset=[
            "District",
        ]
    )
    .dissolve(
        by="District",
        as_index=False,
    )[
        [
            "District",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "District":
                "donor_district",
        }
    )
)


finance["longitude"] = pd.to_numeric(
    finance["longitude"],
    errors="coerce",
)


finance["latitude"] = pd.to_numeric(
    finance["latitude"],
    errors="coerce",
)


finance["has_coordinates"] = (
    finance["longitude"].notna()
    & finance["latitude"].notna()
)


finance["row_id"] = np.arange(
    len(finance)
)


geocoded = finance[
    finance["has_coordinates"]
].copy()


points = gpd.GeoDataFrame(
    geocoded[
        [
            "row_id",
            "longitude",
            "latitude",
        ]
    ].copy(),
    geometry=gpd.points_from_xy(
        geocoded["longitude"],
        geocoded["latitude"],
    ),
    crs="EPSG:4326",
)


points = points.to_crs(
    council_districts.crs
)


spatial_match = (
    gpd.sjoin(
        points,
        council_districts,
        how="left",
        predicate="within",
    )
    .drop_duplicates(
        subset=[
            "row_id",
        ]
    )
)


donor_district_lookup = (
    spatial_match
    .set_index(
        "row_id"
    )[
        "donor_district"
    ]
)


finance["donor_district"] = (
    finance["row_id"]
    .map(
        donor_district_lookup
    )
)


finance["inside_portland"] = (
    finance["has_coordinates"]
    & finance["donor_district"].notna()
)


finance["inside_candidate_district"] = (
    finance["has_coordinates"]
    & finance["donor_district"].notna()
    & finance["donor_district"].eq(
        finance["district"]
    )
)


state = (
    finance[
        "state"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)


finance["state_known"] = (
    state.notna()
    & state.ne("")
    & state.ne("<NA>")
)


finance["inside_oregon"] = (
    finance["state_known"]
    & state.isin(
        [
            "OR",
            "OREGON",
        ]
    )
)


print(
    "Records with coordinates:",
    f"{finance['has_coordinates'].sum():,}",
)

print(
    "Records inside Portland:",
    f"{finance['inside_portland'].sum():,}",
)

print(
    "Records with known state:",
    f"{finance['state_known'].sum():,}",
)

print(
    "Records inside Oregon:",
    f"{finance['inside_oregon'].sum():,}",
)


Records with coordinates: 22,797
Records inside Portland: 17,929
Records with known state: 42,944
Records inside Oregon: 38,873


# PART B — BUILD ALL X VARIABLES

## 11. Scale

In [41]:
candidate_features, scale_features = (
    build_base_features(
        finance,
        profile_keys=PROFILE_KEYS,
        amount_column="amount",
    )
)


scale_features


['total_amount', 'total_record_count', 'mean_amount', 'median_amount']

## 12. Overall bins — all four measures

In [42]:
candidate_features, bin_features = (
    add_bin_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        amount_column="amount",
        bin_column="amount_bin",
        prefix="bin",
    )
)


print(
    "Overall-bin X variables:",
    len(bin_features),
)


Overall-bin X variables: 20


## 13. Cash / In-kind — overall

In [43]:
candidate_features, subtype_features = (
    add_group_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        group_column="subtype_group",
        groups=[
            "cash",
            "in_kind",
        ],
        amount_column="amount",
    )
)


subtype_features


['cash_amount',
 'cash_amount_share',
 'cash_count',
 'cash_count_share',
 'in_kind_amount',
 'in_kind_amount_share',
 'in_kind_count',
 'in_kind_count_share']

### Cash / In-kind × bins — all four measures

For every group × bin we now create:

- amount;
- amount share within the group;
- record count;
- record share within the group.


In [44]:
candidate_features, subtype_bin_features = (
    add_group_bin_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        group_column="subtype_group",
        groups=[
            "cash",
            "in_kind",
        ],
        bin_column="amount_bin",
        amount_column="amount",
    )
)


print(
    "Cash/In-kind bin X variables:",
    len(subtype_bin_features),
)


Cash/In-kind bin X variables: 40


## 14. Individual / Business — overall

In [45]:
candidate_features, entity_features = (
    add_group_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        group_column="entity_group",
        groups=[
            "individual",
            "business",
        ],
        amount_column="amount",
    )
)


entity_features


['individual_amount',
 'individual_amount_share',
 'individual_count',
 'individual_count_share',
 'business_amount',
 'business_amount_share',
 'business_count',
 'business_count_share']

### Individual / Business × bins — all four measures

In [46]:
candidate_features, entity_bin_features = (
    add_group_bin_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        group_column="entity_group",
        groups=[
            "individual",
            "business",
        ],
        bin_column="amount_bin",
        amount_column="amount",
    )
)


print(
    "Individual/Business bin X variables:",
    len(entity_bin_features),
)


Individual/Business bin X variables: 40


## 15. Geography — count and count share only

Geography remains intentionally count-based for this first screen.


In [47]:
candidate_features, same_district_features = (
    add_flag_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        flag_column="inside_candidate_district",
        eligible_column="has_coordinates",
        prefix="same_district",
        amount_column="amount",
    )
)


candidate_features, portland_features = (
    add_flag_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        flag_column="inside_portland",
        eligible_column="has_coordinates",
        prefix="portland",
        amount_column="amount",
    )
)


candidate_features, oregon_features = (
    add_flag_features(
        finance,
        candidate_features,
        profile_keys=PROFILE_KEYS,
        flag_column="inside_oregon",
        eligible_column="state_known",
        prefix="oregon",
        amount_column="amount",
    )
)


geography_features = (
    same_district_features
    + portland_features
    + oregon_features
)


geography_features


['same_district_count',
 'same_district_share',
 'portland_count',
 'portland_share',
 'oregon_count',
 'oregon_share']

### Geography × bins — count and count share

In [48]:
geographic_bin_features = []


geography_settings = [
    (
        "inside_candidate_district",
        "same_district",
    ),
    (
        "inside_portland",
        "portland",
    ),
    (
        "inside_oregon",
        "oregon",
    ),
]


for flag_column, prefix in geography_settings:

    candidate_features, new_features = (
        add_flag_bin_features(
            finance,
            candidate_features,
            profile_keys=PROFILE_KEYS,
            flag_column=flag_column,
            prefix=prefix,
            bin_column="amount_bin",
            amount_column="amount",
        )
    )


    geographic_bin_features.extend(
        new_features
    )


print(
    "Geographic-bin X variables:",
    len(geographic_bin_features),
)


Geographic-bin X variables: 30


## 16. Assemble the complete X list

In [49]:
candidate_features = (
    candidate_features
    .merge(
        candidate_names,
        on=PROFILE_KEYS,
        how="left",
        validate="one_to_one",
    )
)


FEATURE_GROUPS = {
    "scale":
        scale_features,

    "overall_bins":
        bin_features,

    "cash_in_kind":
        subtype_features,

    "cash_in_kind_bins":
        subtype_bin_features,

    "individual_business":
        entity_features,

    "individual_business_bins":
        entity_bin_features,

    "geography":
        geography_features,

    "geography_bins":
        geographic_bin_features,
}


X_VARIABLES = []


for family, variables in FEATURE_GROUPS.items():

    X_VARIABLES.extend(
        variables
    )


X_VARIABLES = list(
    dict.fromkeys(
        X_VARIABLES
    )
)


FEATURE_FAMILY = {}


for family, variables in FEATURE_GROUPS.items():

    for variable in variables:

        FEATURE_FAMILY[
            variable
        ] = family


print(
    "Total X variables:",
    len(X_VARIABLES),
)


Total X variables: 156


## 17. X-variable inventory

This table is the reference for the rest of the notebook.

It shows:

- feature name;
- family;
- measure type.


In [50]:
def simple_measure_name(
    feature
):

    if "amount_share" in feature:
        return "dollar share"

    if "count_share" in feature:
        return "record share"

    if feature.endswith("_share"):
        return "record share"

    if "amount" in feature:
        return "dollar amount"

    if "count" in feature:
        return "record count"

    return "other"


feature_inventory = pd.DataFrame(
    [
        {
            "feature":
                variable,

            "family":
                FEATURE_FAMILY[
                    variable
                ],

            "measure":
                simple_measure_name(
                    variable
                ),
        }
        for variable
        in X_VARIABLES
    ]
)


display(
    feature_inventory
    .sort_values(
        [
            "family",
            "measure",
            "feature",
        ]
    )
    .reset_index(
        drop=True
    )
)


,feature,family,measure
0,cash_amount,cash_in_kind,dollar amount
1,in_kind_amount,cash_in_kind,dollar amount
2,cash_amount_share,cash_in_kind,dollar share
3,in_kind_amount_share,cash_in_kind,dollar share
4,cash_count,cash_in_kind,record count
...,...,...,...
151,bin_count_share_small,overall_bins,record share
152,mean_amount,scale,dollar amount
153,median_amount,scale,dollar amount
154,total_amount,scale,dollar amount


## 18. Quick QA

In [51]:
overall_count_share_columns = [
    f"bin_count_share_{bin_name}"
    for bin_name
    in BIN_LABELS
]


overall_count_share_sum = (
    candidate_features[
        overall_count_share_columns
    ]
    .sum(
        axis=1
    )
)


print(
    "Overall record-share bins sum to 1:",
    np.isclose(
        overall_count_share_sum,
        1.0,
    ).all(),
)


overall_amount_share_columns = [
    f"bin_amount_share_{bin_name}"
    for bin_name
    in BIN_LABELS
]


overall_amount_share_sum = (
    candidate_features[
        overall_amount_share_columns
    ]
    .sum(
        axis=1
    )
)


print(
    "Overall dollar-share bins sum to 1:",
    np.isclose(
        overall_amount_share_sum,
        1.0,
    ).all(),
)


print(
    "same district <= Portland:",
    (
        candidate_features[
            "same_district_count"
        ]
        <= candidate_features[
            "portland_count"
        ]
    ).all(),
)


Overall record-share bins sum to 1: True
Overall dollar-share bins sum to 1: True
same district <= Portland: True


# PART C — CORRELATIONS

## 19. Merge finance features with ballot support

In [52]:
# ---------------------------------------------------------------
# Full analysis table for the selected year
# ---------------------------------------------------------------

analysis_all_districts = (
    support
    .merge(
        candidate_features.drop(
            columns=[
                "canonical_candidate",
            ]
        ),
        on=[
            "year",
            "district",
            "candidate_key",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------------
# Main analysis sample
# ---------------------------------------------------------------

if DISTRICT == "all":

    analysis = (
        analysis_all_districts
        .copy()
    )

else:

    analysis = (
        analysis_all_districts[
            analysis_all_districts[
                "district"
            ].eq(
                DISTRICT
            )
        ]
        .copy()
    )


print(
    "Main analysis district:",
    DISTRICT,
)


print(
    "Candidates in main analysis:",
    len(
        analysis
    ),
)


print(
    "Candidates with matched fundraising:",
    analysis[
        "total_amount"
    ].notna().sum(),
)


print(
    "Candidates available for independent district races:",
    len(
        analysis_all_districts
    ),
)


Main analysis district: all
Candidates in main analysis: 98
Candidates with matched fundraising: 65
Candidates available for independent district races: 98


## 20. Run all correlations

The helper still calculates p-values and FDR-adjusted p-values for the
exported full result, but the main exploratory tables below focus on:

- `feature`
- `family`
- `measure`
- `n`
- Pearson `r`
- Spearman `rho`


In [53]:
correlations = calculate_correlations(
    analysis,
    x_variables=X_VARIABLES,
    outcomes=OUTCOMES,
    feature_family=FEATURE_FAMILY,
)


correlations.insert(
    0,
    "source",
    "OpenElections contributions",
)


correlations.insert(
    1,
    "year_filter",
    (
        "all_available"
        if YEAR is None
        else YEAR
    ),
)


correlations.insert(
    2,
    "district_filter",
    (
        "all"
        if DISTRICT is None
        else DISTRICT
    ),
)


print(
    "Correlation rows:",
    len(correlations),
)


Correlation rows: 280


# 21. Overall correlation race

First question:

> **Which X variables are most strongly correlated with each electoral outcome
> when all selected candidates are analyzed together?**

The table is ordered by `|Pearson r|`.


In [54]:
for outcome in OUTCOMES:

    table = make_correlation_race(
        correlations,
        outcome=outcome,
        top_n=TOP_N,
    )


    print()
    print(
        f"OVERALL CORRELATION RACE — {outcome}"
    )


    display(
        table[
            [
                "rank",
                "feature",
                "family",
                "measure",
                "n",
                "pearson_r",
                "spearman_r",
            ]
        ]
        .round(
            {
                "pearson_r": 3,
                "spearman_r": 3,
            }
        )
    )



OVERALL CORRELATION RACE — mentions


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,bin_amount_small,overall_bins,dollar amount,65,0.803,0.819
1,2,individual_bin_amount_small,individual_business_bins,dollar amount,65,0.802,0.819
2,3,cash_bin_amount_small,cash_in_kind_bins,dollar amount,65,0.802,0.820
3,4,cash_bin_count_small,cash_in_kind_bins,record count,65,0.801,0.818
4,5,individual_bin_count_small,individual_business_bins,record count,65,0.801,0.816
5,6,bin_count_small,overall_bins,record count,65,0.801,0.819
6,7,oregon_bin_count_small,geography_bins,record count,65,0.796,0.803
7,8,total_amount,scale,dollar amount,65,0.779,0.800
8,9,cash_amount,cash_in_kind,dollar amount,65,0.767,0.796
9,10,individual_amount,individual_business,dollar amount,65,0.766,0.801



OVERALL CORRELATION RACE — first_place_votes


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,individual_bin_amount_small,individual_business_bins,dollar amount,65,0.793,0.774
1,2,cash_bin_amount_small,cash_in_kind_bins,dollar amount,65,0.793,0.777
2,3,bin_amount_small,overall_bins,dollar amount,65,0.793,0.774
3,4,oregon_bin_count_small,geography_bins,record count,65,0.791,0.771
4,5,individual_bin_count_small,individual_business_bins,record count,65,0.781,0.772
5,6,cash_bin_count_small,cash_in_kind_bins,record count,65,0.781,0.776
6,7,bin_count_small,overall_bins,record count,65,0.780,0.773
7,8,total_amount,scale,dollar amount,65,0.752,0.764
8,9,individual_amount,individual_business,dollar amount,65,0.746,0.764
9,10,cash_amount,cash_in_kind,dollar amount,65,0.745,0.759


# 22. Best feature in each family

The overall race can contain several versions of nearly the same concept.

Here we keep only the strongest X variable from each family.


In [55]:
family_best = (
    correlations
    .sort_values(
        "abs_pearson_r",
        ascending=False,
    )
    .groupby(
        [
            "outcome",
            "family",
        ],
        as_index=False,
    )
    .first()
)


for outcome in OUTCOMES:

    table = (
        family_best[
            family_best[
                "outcome"
            ].eq(
                outcome
            )
        ]
        .sort_values(
            "abs_pearson_r",
            ascending=False,
        )
        .copy()
        .reset_index(
            drop=True
        )
    )


    table.insert(
        0,
        "rank",
        np.arange(
            1,
            len(table) + 1,
        ),
    )


    print()
    print(
        f"BEST FEATURE BY FAMILY — {outcome}"
    )


    display(
        table[
            [
                "rank",
                "family",
                "feature",
                "measure",
                "n",
                "pearson_r",
                "spearman_r",
            ]
        ]
        .round(
            {
                "pearson_r": 3,
                "spearman_r": 3,
            }
        )
    )



BEST FEATURE BY FAMILY — mentions


,rank,family,feature,measure,n,pearson_r,spearman_r
0,1,overall_bins,bin_amount_small,dollar amount,65,0.803,0.819
1,2,individual_business_bins,individual_bin_amount_small,dollar amount,65,0.802,0.819
2,3,cash_in_kind_bins,cash_bin_amount_small,dollar amount,65,0.802,0.820
3,4,geography_bins,oregon_bin_count_small,record count,65,0.796,0.803
4,5,scale,total_amount,dollar amount,65,0.779,0.800
5,6,cash_in_kind,cash_amount,dollar amount,65,0.767,0.796
6,7,individual_business,individual_amount,dollar amount,65,0.766,0.801
7,8,geography,oregon_count,record count,65,0.723,0.748



BEST FEATURE BY FAMILY — first_place_votes


,rank,family,feature,measure,n,pearson_r,spearman_r
0,1,individual_business_bins,individual_bin_amount_small,dollar amount,65,0.793,0.774
1,2,cash_in_kind_bins,cash_bin_amount_small,dollar amount,65,0.793,0.777
2,3,overall_bins,bin_amount_small,dollar amount,65,0.793,0.774
3,4,geography_bins,oregon_bin_count_small,record count,65,0.791,0.771
4,5,scale,total_amount,dollar amount,65,0.752,0.764
5,6,individual_business,individual_amount,dollar amount,65,0.746,0.764
6,7,cash_in_kind,cash_amount,dollar amount,65,0.745,0.759
7,8,geography,oregon_count,record count,65,0.684,0.721


# 23. Best feature by family × measure

Within each family, this keeps the strongest available:

- dollar amount;
- dollar share;
- record count;
- record share.


In [56]:
family_measure_best = (
    correlations
    .sort_values(
        "abs_pearson_r",
        ascending=False,
    )
    .groupby(
        [
            "outcome",
            "family",
            "measure",
        ],
        as_index=False,
    )
    .first()
)


for outcome in OUTCOMES:

    table = (
        family_measure_best[
            family_measure_best[
                "outcome"
            ].eq(
                outcome
            )
        ]
        .sort_values(
            "abs_pearson_r",
            ascending=False,
        )
        .copy()
        .reset_index(
            drop=True
        )
    )


    table.insert(
        0,
        "rank",
        np.arange(
            1,
            len(table) + 1,
        ),
    )


    print()
    print(
        f"BEST FEATURE BY FAMILY × MEASURE — {outcome}"
    )


    display(
        table[
            [
                "rank",
                "family",
                "measure",
                "feature",
                "n",
                "pearson_r",
                "spearman_r",
            ]
        ]
        .round(
            {
                "pearson_r": 3,
                "spearman_r": 3,
            }
        )
    )



BEST FEATURE BY FAMILY × MEASURE — mentions


,rank,family,measure,feature,n,pearson_r,spearman_r
0,1,overall_bins,dollar amount,bin_amount_small,65,0.803,0.819
1,2,individual_business_bins,dollar amount,individual_bin_amount_small,65,0.802,0.819
2,3,cash_in_kind_bins,dollar amount,cash_bin_amount_small,65,0.802,0.820
3,4,cash_in_kind_bins,record count,cash_bin_count_small,65,0.801,0.818
4,5,individual_business_bins,record count,individual_bin_count_small,65,0.801,0.816
5,6,overall_bins,record count,bin_count_small,65,0.801,0.819
6,7,geography_bins,record count,oregon_bin_count_small,65,0.796,0.803
7,8,scale,dollar amount,total_amount,65,0.779,0.800
8,9,cash_in_kind,dollar amount,cash_amount,65,0.767,0.796
9,10,individual_business,dollar amount,individual_amount,65,0.766,0.801



BEST FEATURE BY FAMILY × MEASURE — first_place_votes


,rank,family,measure,feature,n,pearson_r,spearman_r
0,1,individual_business_bins,dollar amount,individual_bin_amount_small,65,0.793,0.774
1,2,cash_in_kind_bins,dollar amount,cash_bin_amount_small,65,0.793,0.777
2,3,overall_bins,dollar amount,bin_amount_small,65,0.793,0.774
3,4,geography_bins,record count,oregon_bin_count_small,65,0.791,0.771
4,5,individual_business_bins,record count,individual_bin_count_small,65,0.781,0.772
5,6,cash_in_kind_bins,record count,cash_bin_count_small,65,0.781,0.776
6,7,overall_bins,record count,bin_count_small,65,0.780,0.773
7,8,scale,dollar amount,total_amount,65,0.752,0.764
8,9,individual_business,dollar amount,individual_amount,65,0.746,0.764
9,10,cash_in_kind,dollar amount,cash_amount,65,0.745,0.759


# 24. Correlations by district

These races are **independent from the `DISTRICT` setting above**.

For the selected year, we go back to the full candidate sample and calculate
the same correlation screen separately for:

- District 1
- District 2
- District 3
- District 4

So:

```python
DISTRICT = "all"
```

and:

```python
DISTRICT = 4
```

can change the **main race**, but they do not change how the D1–D4 races are calculated.

`MIN_DISTRICT_N` is only used to avoid showing district correlations based on
very small samples.


In [57]:
district_correlations = (
    calculate_correlations_by_district(
        analysis_all_districts,
        x_variables=X_VARIABLES,
        outcomes=OUTCOMES,
        feature_family=FEATURE_FAMILY,
        min_n=MIN_DISTRICT_N,
    )
)


DISTRICTS = sorted(
    district_correlations[
        "district"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


print(
    "Districts:",
    DISTRICTS,
)

print(
    "Minimum district n:",
    MIN_DISTRICT_N,
)


Districts: [1, 2, 3, 4]
Minimum district n: 8


## 25. District correlation races

Each district gets the same race as the pooled sample.


In [58]:
for outcome in OUTCOMES:

    for district in DISTRICTS:

        table = make_district_race(
            district_correlations,
            outcome=outcome,
            district=district,
            top_n=DISTRICT_TOP_N,
        )


        print()
        print(
            f"DISTRICT {district} — {outcome}"
        )


        display(
            table[
                [
                    "rank",
                    "feature",
                    "family",
                    "measure",
                    "n",
                    "pearson_r",
                    "spearman_r",
                ]
            ]
            .round(
                {
                    "pearson_r": 3,
                    "spearman_r": 3,
                }
            )
        )



DISTRICT 1 — mentions


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,total_amount,scale,dollar amount,13,0.919,0.923
1,2,cash_amount,cash_in_kind,dollar amount,13,0.904,0.923
2,3,oregon_count,geography,record count,13,0.899,0.934
3,4,total_record_count,scale,record count,13,0.898,0.934
4,5,bin_amount_micro,overall_bins,dollar amount,13,0.897,0.951
5,6,cash_bin_amount_micro,cash_in_kind_bins,dollar amount,13,0.897,0.951
6,7,individual_bin_amount_micro,individual_business_bins,dollar amount,13,0.897,0.951
7,8,cash_count,cash_in_kind,record count,13,0.897,0.934
8,9,individual_count,individual_business,record count,13,0.895,0.934
9,10,oregon_bin_count_medium,geography_bins,record count,13,0.895,0.893



DISTRICT 2 — mentions


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,bin_count_small,overall_bins,record count,20,0.823,0.827
1,2,cash_bin_count_small,cash_in_kind_bins,record count,20,0.822,0.826
2,3,individual_bin_count_small,individual_business_bins,record count,20,0.821,0.826
3,4,bin_amount_small,overall_bins,dollar amount,20,0.794,0.830
4,5,cash_bin_amount_small,cash_in_kind_bins,dollar amount,20,0.792,0.829
5,6,individual_bin_amount_small,individual_business_bins,dollar amount,20,0.790,0.829
6,7,oregon_bin_count_small,geography_bins,record count,20,0.779,0.822
7,8,total_amount,scale,dollar amount,20,0.731,0.774
8,9,cash_amount,cash_in_kind,dollar amount,20,0.714,0.789
9,10,cash_count,cash_in_kind,record count,20,0.710,0.744



DISTRICT 3 — mentions


...

,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,in_kind_bin_amount_share_mega,cash_in_kind_bins,dollar share,8,0.670,0.577
1,2,in_kind_bin_count_share_mega,cash_in_kind_bins,record share,8,0.670,0.577
2,3,bin_amount_small,overall_bins,dollar amount,20,0.661,0.783
3,4,cash_bin_amount_small,cash_in_kind_bins,dollar amount,20,0.661,0.785
4,5,oregon_bin_count_small,geography_bins,record count,20,0.661,0.801
5,6,individual_bin_amount_small,individual_business_bins,dollar amount,20,0.660,0.785
6,7,cash_bin_count_small,cash_in_kind_bins,record count,20,0.659,0.785
7,8,individual_bin_count_small,individual_business_bins,record count,20,0.658,0.785
8,9,bin_count_small,overall_bins,record count,20,0.658,0.783
9,10,total_amount,scale,dollar amount,20,0.651,0.753



DISTRICT 3 — first_place_votes


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,cash_bin_amount_small,cash_in_kind_bins,dollar amount,15,0.945,0.796
1,2,individual_bin_amount_small,individual_business_bins,dollar amount,15,0.944,0.789
2,3,bin_amount_small,overall_bins,dollar amount,15,0.942,0.796
3,4,individual_amount,individual_business,dollar amount,15,0.939,0.771
4,5,cash_amount,cash_in_kind,dollar amount,15,0.938,0.761
5,6,total_amount,scale,dollar amount,15,0.933,0.761
6,7,oregon_bin_count_small,geography_bins,record count,15,0.927,0.761
7,8,same_district_bin_count_medium,geography_bins,record count,15,0.925,0.831
8,9,cash_bin_count_small,cash_in_kind_bins,record count,15,0.914,0.792
9,10,individual_bin_count_small,individual_business_bins,record count,15,0.913,0.785



DISTRICT 4 — first_place_votes


,rank,feature,family,measure,n,pearson_r,spearman_r
0,1,individual_bin_amount_small,individual_business_bins,dollar amount,17,0.822,0.665
1,2,bin_amount_small,overall_bins,dollar amount,17,0.821,0.665
2,3,cash_bin_amount_small,cash_in_kind_bins,dollar amount,17,0.819,0.665
3,4,individual_bin_count_small,individual_business_bins,record count,17,0.791,0.677
4,5,bin_count_small,overall_bins,record count,17,0.790,0.677
5,6,cash_bin_count_small,cash_in_kind_bins,record count,17,0.788,0.677
6,7,oregon_bin_count_small,geography_bins,record count,17,0.784,0.662
7,8,total_amount,scale,dollar amount,17,0.777,0.701
8,9,portland_bin_count_small,geography_bins,record count,17,0.766,0.596
9,10,individual_amount,individual_business,dollar amount,17,0.765,0.711


# 26. Pooled vs. districts

This is the main district-comparison table.

For the strongest pooled X variables we display:

`pooled_r | D1_r | D2_r | D3_r | D4_r`

This lets us see immediately whether a relationship is similar across districts
or driven by only one or two of them.


In [59]:
district_comparison_tables = {}


for outcome in OUTCOMES:

    table = (
        compare_top_features_across_districts(
            pooled_correlations=correlations,
            district_correlations=district_correlations,
            outcome=outcome,
            top_n=COMPARE_TOP_N,
        )
    )


    district_comparison_tables[
        outcome
    ] = table


    print()
    print(
        f"POOLED VS DISTRICTS — {outcome}"
    )


    display(
        table.round(
            3
        )
    )



POOLED VS DISTRICTS — mentions


,pooled_rank,feature,family,measure,pooled_r,D1_r,D2_r,D3_r,D4_r
0,1,bin_amount_small,overall_bins,dollar amount,0.803,0.860,0.794,0.941,0.717
1,2,individual_bin_amount_small,individual_business_bins,dollar amount,0.802,0.856,0.790,0.940,0.718
2,3,cash_bin_amount_small,cash_in_kind_bins,dollar amount,0.802,0.858,0.792,0.941,0.715
3,4,cash_bin_count_small,cash_in_kind_bins,record count,0.801,0.859,0.822,0.921,0.698
4,5,individual_bin_count_small,individual_business_bins,record count,0.801,0.857,0.821,0.920,0.701
5,6,bin_count_small,overall_bins,record count,0.801,0.860,0.823,0.921,0.700
6,7,oregon_bin_count_small,geography_bins,record count,0.796,0.871,0.779,0.932,0.686
7,8,total_amount,scale,dollar amount,0.779,0.919,0.731,0.951,0.722
8,9,cash_amount,cash_in_kind,dollar amount,0.767,0.904,0.714,0.951,0.685
9,10,individual_amount,individual_business,dollar amount,0.766,0.893,0.705,0.949,0.697



POOLED VS DISTRICTS — first_place_votes


,pooled_rank,feature,family,measure,pooled_r,D1_r,D2_r,D3_r,D4_r
0,1,individual_bin_amount_small,individual_business_bins,dollar amount,0.793,0.665,0.660,0.944,0.822
1,2,cash_bin_amount_small,cash_in_kind_bins,dollar amount,0.793,0.667,0.661,0.945,0.819
2,3,bin_amount_small,overall_bins,dollar amount,0.793,0.675,0.661,0.942,0.821
3,4,oregon_bin_count_small,geography_bins,record count,0.791,0.706,0.661,0.927,0.784
4,5,individual_bin_count_small,individual_business_bins,record count,0.781,0.675,0.658,0.913,0.791
5,6,cash_bin_count_small,cash_in_kind_bins,record count,0.781,0.677,0.659,0.914,0.788
6,7,bin_count_small,overall_bins,record count,0.780,0.685,0.658,0.911,0.790
7,8,total_amount,scale,dollar amount,0.752,0.758,0.651,0.933,0.777
8,9,individual_amount,individual_business,dollar amount,0.746,0.710,0.634,0.939,0.765
9,10,cash_amount,cash_in_kind,dollar amount,0.745,0.725,0.640,0.938,0.751


# 27. Export

The full correlation CSV still contains p-values and FDR-adjusted p-values
for reference, even though we do not foreground them in the notebook.


In [60]:
year_label = (
    "all_available_years"
    if YEAR is None
    else str(YEAR)
)


district_label = (
    "all_districts"
    if DISTRICT == "all"
    else f"district_{DISTRICT}"
)


scope_label = (
    f"{year_label}_{district_label}"
)


OUTPUT_DIR = (
    PROCESSED
    / "finance_analysis"
    / "correlation_screens"
    / "openelections_contributions"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


outputs = {
    "candidate_features":
        candidate_features,

    "candidate_features_with_support":
        analysis,

    "correlations_full":
        correlations,

    "family_best":
        family_best,

    "family_measure_best":
        family_measure_best,

    "feature_inventory":
        feature_inventory,

    "district_correlations":
        district_correlations,
}


for name, table in outputs.items():

    path = (
        OUTPUT_DIR
        / f"{scope_label}_{name}.csv"
    )


    table.to_csv(
        path,
        index=False,
    )


    print(
        "SAVED:",
        path,
    )


# Save the simple pooled-vs-district comparison for each outcome.
for outcome, table in district_comparison_tables.items():

    path = (
        OUTPUT_DIR
        / (
            f"{scope_label}_"
            f"{outcome}_pooled_vs_districts.csv"
        )
    )


    table.to_csv(
        path,
        index=False,
    )


    print(
        "SAVED:",
        path,
    )


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/correlation_screens/openelections_contributions/2024_all_districts_candidate_features.csv
SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/correlation_screens/openelections_contributions/2024_all_districts_candidate_features_with_support.csv
SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/correlation_screens/openelections_contributions/2024_all_districts_correlations_full.csv
SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/correlation_screens/openelections_contributions/2024_all_districts_family_best.csv
SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/correlation_screens/openelections_contributions/2024_all_districts_fam

# 28. Research notes

## Scale
- ...

## Overall bins
- ...

## Cash / In-kind
- ...

## Individual / Business
- ...

## Geography
- ...

## Dollar-level vs. contribution-record-level patterns
- ...

## Variables to carry into regression models
- ...
